# Fine-tuning Qwen2.5-0.5B-Instruct with QLoRA

**Task:** Structured Information Extraction from unstructured text

**Run this notebook on Google Colab with a T4 GPU runtime.**

Steps:
1. Install dependencies
2. Upload dataset
3. Train with QLoRA
4. Evaluate base vs fine-tuned
5. Download LoRA adapter

In [ ]:
# Step 0: Install dependencies
!pip install -q torch transformers peft bitsandbytes trl datasets accelerate scipy

In [ ]:
# Step 0.5: Clone repo (if running on Colab)
import os
if not os.path.exists('fine_tuning_SLM'):
    !git clone https://github.com/v3rmxni7/fine_tuning_SLM.git
    os.chdir('fine_tuning_SLM')
else:
    os.chdir('fine_tuning_SLM')

# Verify GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Step 1: Generate dataset (if not already generated)
if not os.path.exists('data/processed/train.json'):
    !python data/generate_dataset.py
else:
    print('Dataset already exists.')

import json
with open('data/processed/train.json') as f:
    train_data = json.load(f)
print(f'Training samples: {len(train_data)}')
print(f'Sample: {json.dumps(train_data[0], indent=2)[:300]}')

In [ ]:
# Step 2: Load model and tokenizer
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)
print('Model loaded.')

In [ ]:
# Step 3: Attach LoRA adapters
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'],
    task_type='CAUSAL_LM',
    bias='none',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Step 4: Prepare dataset for training
from datasets import Dataset
from src.formatting import format_chat_messages

def prepare_data(samples):
    formatted = []
    for sample in samples:
        category = sample.get('category', 'resume')
        messages = format_chat_messages(sample, category)
        text = tokenizer.apply_chat_template(messages, tokenize=False)
        formatted.append({'text': text})
    return Dataset.from_list(formatted)

with open('data/processed/train.json') as f:
    train_samples = json.load(f)
with open('data/processed/val.json') as f:
    val_samples = json.load(f)

train_dataset = prepare_data(train_samples)
val_dataset = prepare_data(val_samples)

print(f'Train: {len(train_dataset)}, Val: {len(val_dataset)}')
print(f'\nSample formatted text:\n{train_dataset[0]["text"][:500]}')

In [ ]:
# Step 5: Train
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir='outputs',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=10,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=50,
    save_strategy='epoch',
    fp16=True,
    report_to='none',
    remove_unused_columns=False,
    optim='paged_adamw_8bit',
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    max_seq_length=512,
)

print('Starting training...')
trainer.train()

In [ ]:
# Step 6: Save LoRA adapter
ADAPTER_DIR = 'outputs/lora_adapter'
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f'Adapter saved to {ADAPTER_DIR}')

# List saved files
for f in os.listdir(ADAPTER_DIR):
    size_mb = os.path.getsize(os.path.join(ADAPTER_DIR, f)) / 1024 / 1024
    print(f'  {f}: {size_mb:.2f} MB')

In [ ]:
# Step 7: Quick inference test (base vs fine-tuned)
from peft import PeftModel

test_input = 'Sarah Chen is a senior data engineer with 5 years of experience in Apache Spark, Python, and AWS. She holds a Masters in CS from Stanford.'

# Format prompt
from src.formatting import format_inference_messages
messages = format_inference_messages(test_input, 'resume')

def generate(model, messages):
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=512)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=256, do_sample=False, pad_token_id=tokenizer.pad_token_id)
    input_len = inputs['input_ids'].shape[1]
    return tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()

# Fine-tuned output
ft_output = generate(model, messages)
print('Fine-tuned output:')
print(ft_output)

# Base model output (reload without adapter)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
base_output = generate(base_model, messages)
print('\nBase model output:')
print(base_output)

In [ ]:
# Step 8: Run full evaluation on test set
from src.inference import InferencePipeline

with open('data/processed/test.json') as f:
    test_data = json.load(f)

print(f'Evaluating on {len(test_data)} test samples...')

# 8a: Base model (no guardrails)
print('\n--- Evaluating Base Model ---')
base_pipeline = InferencePipeline(use_finetuned=False, use_guardrails=False)
for sample in test_data:
    base_pipeline.extract(sample['input'], category=sample['category'])
base_pipeline.save_logs('evaluation/logs_base.json')

# 8b: Fine-tuned (no guardrails)
print('\n--- Evaluating Fine-tuned Model ---')
ft_pipeline = InferencePipeline(use_finetuned=True, use_guardrails=False)
for sample in test_data:
    ft_pipeline.extract(sample['input'], category=sample['category'])
ft_pipeline.save_logs('evaluation/logs_finetuned.json')

# 8c: Fine-tuned + Guardrails
print('\n--- Evaluating Fine-tuned + Guardrails ---')
ftg_pipeline = InferencePipeline(use_finetuned=True, use_guardrails=True)
for sample in test_data:
    ftg_pipeline.extract(sample['input'], category=sample['category'])
ftg_pipeline.save_logs('evaluation/logs_finetuned_guardrails.json')

In [ ]:
# Step 9: Print evaluation results
from evaluation.eval import run_evaluation, print_comparison_table, save_results, load_test_data

test_data = load_test_data()
results = []

for log_path, label in [
    ('evaluation/logs_base.json', 'Base Model'),
    ('evaluation/logs_finetuned.json', 'Fine-tuned'),
    ('evaluation/logs_finetuned_guardrails.json', 'Fine-tuned + Guardrails'),
]:
    if os.path.exists(log_path):
        metrics = run_evaluation(log_path, label, test_data)
        results.append(metrics)

print_comparison_table(results)
save_results(results)

In [ ]:
# Step 10: Download adapter (for Colab)
# Zip the adapter for easy download
!zip -r lora_adapter.zip outputs/lora_adapter/
print('Download lora_adapter.zip from the file browser on the left.')

# Optional: Push to HuggingFace
# from huggingface_hub import login
# login(token='YOUR_HF_TOKEN')
# model.push_to_hub('your-username/qwen2.5-0.5b-extraction-lora')
# tokenizer.push_to_hub('your-username/qwen2.5-0.5b-extraction-lora')